# P1 — Stage A ModernBERT + LoRA Training on Kaggle T4

**Before running:**
1. Set accelerator to **GPU T4 x2** in Notebook settings → Accelerator
2. Set Internet access to **ON** (needed to clone repo and download model/datasets)
3. Run all cells in order

**Outputs saved to:**  (adapter zip, ONNX zip, manifest JSON)

In [ ]:
# Cell 1 — Verify GPU
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# Cell 2 — Clone / update repository
import os
REPO_URL = "https://github.com/Priyrajsinh/Hybrid-LLM-Jailbreak-Detector.git"
WORK_DIR = "/kaggle/working/Hybrid-LLM-Jailbreak-Detector"

if not os.path.exists(WORK_DIR):
    subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=True)

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())

In [ ]:
# Cell 3 — Install dependencies (including optimum for ONNX export)
subprocess.run(
    ["pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
    cwd=WORK_DIR,
)
# optimum is needed for ONNX export (not in requirements.txt — GPU-only dep)
subprocess.run(
    ["pip", "install", "-q", "optimum[onnxruntime]"],
    check=True,
    cwd=WORK_DIR,
)
print("Dependencies installed.")

In [ ]:
# Cell 4 — Build dataset splits from full 5-source datasets
# ALWAYS regenerate — never use the dev parquets committed to the repo.
# Wipes any existing data/processed/ so the pipeline always runs fresh.
import shutil, pathlib

proc_dir = pathlib.Path("data/processed")
if proc_dir.exists():
    shutil.rmtree(proc_dir)
    print("Removed existing data/processed/ (dev data from repo clone).")

subprocess.run(
    ["python", "-m", "src.data.pipeline", "--config", "config/config.yaml"],
    check=True,
    cwd=WORK_DIR,
)
print("Dataset splits built.")

# Print actual row counts for the manifest
import pandas as pd
n_train = len(pd.read_parquet("data/processed/train.parquet"))
n_val   = len(pd.read_parquet("data/processed/val.parquet"))
n_test  = len(pd.read_parquet("data/processed/test.parquet"))
print(f"train={n_train}  val={n_val}  test={n_test}")

In [ ]:
# Cell 5 — Run Stage A LoRA training
subprocess.run(
    [
        "python", "-m", "src.training.train",
        "--config", "config/config.yaml",
    ],
    check=True,
    cwd=WORK_DIR,
)
print("Training complete.")

In [ ]:
# Cell 6 — Export merged model to ONNX via subprocess (avoids import-path issues)
subprocess.run(
    [
        "python", "-m", "src.training.export_onnx",
        "models/stage_a_merged",
        "models/stage_a_onnx",
        "config/config.yaml",
    ],
    check=True,
    cwd=WORK_DIR,
)
print("ONNX export complete.")

In [ ]:
# Cell 7 — Save model manifest with real training metrics
import json, pathlib, pandas as pd
from src.config import load_config
from src.training.manifest import build_manifest, save_manifest

cfg = load_config("config/config.yaml")

# Read actual training sample count from parquet (never trust MLflow params alone)
n_train = len(pd.read_parquet("data/processed/train.parquet"))

# Read best_val_f1 from MLflow
try:
    import mlflow
    client = mlflow.tracking.MlflowClient()
    exp = client.get_experiment_by_name("p1-hybrid-jailbreak")
    runs = client.search_runs(exp.experiment_id, order_by=["metrics.best_val_f1 DESC"])
    best_f1 = float(runs[0].data.metrics.get("best_val_f1", 0.0)) if runs else 0.0
    # Verify n_train from MLflow param matches parquet (sanity check)
    mlflow_n = int(runs[0].data.params.get("n_train", 0)) if runs else 0
    if mlflow_n > 0 and mlflow_n != n_train:
        print(f"WARNING: MLflow n_train={mlflow_n} != parquet n_train={n_train}")
except Exception as e:
    print(f"MLflow read failed: {e}")
    best_f1 = 0.0

onnx_path = pathlib.Path("models/stage_a_onnx")
manifest = build_manifest(
    config=cfg,
    training_samples=n_train,
    epochs_completed=int(cfg["training"]["epochs"]),
    best_val_f1=best_f1,
    training_device="T4",
    onnx_exported=onnx_path.exists(),
    onnx_speedup_ratio=1.0,
)
save_manifest(manifest, "models/manifests/stage_a_manifest.json")
print(json.dumps(manifest, indent=2))

In [ ]:
# Cell 8 — Zip outputs for download
import shutil, os
shutil.make_archive("/kaggle/working/stage_a_adapter", "zip", "models/stage_a_adapter")
shutil.make_archive("/kaggle/working/stage_a_onnx",    "zip", "models/stage_a_onnx")
shutil.copy("models/manifests/stage_a_manifest.json", "/kaggle/working/")

for f in ["/kaggle/working/stage_a_adapter.zip",
          "/kaggle/working/stage_a_onnx.zip",
          "/kaggle/working/stage_a_manifest.json"]:
    size = os.path.getsize(f) / 1024 / 1024
    print(f"{f}: {size:.1f} MB")